# 04_3 — Entrenamiento CatBoost Residual

Segundo modelo tabular para el snapshot pipeline: `CatBoostRegressor` sobre el target residual `label - price_yes`.

La idea no es tocar el preprocesamiento ya limpio, sino cambiar el ángulo del modelo:
- usa las **features tabulares existentes**
- aprovecha **categoría nativa** sin OHE
- usa el **embedding de texto completo** (sin PCA)
- predice **residual respecto al mercado**, y luego calibra probabilidades

Este notebook **no reentrena** por default. Carga artefactos guardados en `data/models/catboost_residual/`.

In [1]:
import sys
import json
import subprocess
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.metrics import roc_curve, precision_recall_curve, auc
from catboost import CatBoostRegressor, Pool

ROOT = Path('..').resolve()
sys.path.insert(0, str(ROOT))

from src.config import load_config
from src.model.dataset import PolymarketDataset
from src.model.splits import build_dataset_split
from src.model.calibration import ProbabilityCalibrator
from src.model.metrics import clip_probabilities_from_residual

cfg = load_config(str(ROOT / 'config' / 'config.yaml'))
PROCESSED = ROOT / cfg['data']['processed_dir']
SAVE_DIR = ROOT / 'data' / 'models' / 'catboost_residual'

RUN_TRAINING = False  # Cambiar a True para reentrenar

print('Root:', ROOT)
print('Processed:', PROCESSED)
print('Save dir:', SAVE_DIR)


def build_catboost_matrix(numerical, categories, text_embeddings):
    num_dim = numerical.shape[1]
    text_dim = text_embeddings.shape[1]
    matrix = np.empty((len(numerical), num_dim + 1 + text_dim), dtype=object)
    matrix[:, :num_dim] = numerical.astype(np.float32)
    matrix[:, num_dim] = categories.astype(str)
    matrix[:, num_dim + 1:] = text_embeddings.astype(np.float32)
    return matrix

Root: /Users/andres/Documents/ITAM/octavo_semestre/mineria_y_analisis/proyecto_polymarket
Processed: /Users/andres/Documents/ITAM/octavo_semestre/mineria_y_analisis/proyecto_polymarket/data/processed
Save dir: /Users/andres/Documents/ITAM/octavo_semestre/mineria_y_analisis/proyecto_polymarket/data/models/catboost_residual


## 1. Cargar dataset y estadísticas

In [2]:
dataset = PolymarketDataset.from_numpy_dir(str(PROCESSED))

n = len(dataset)
n_pos = int(dataset.labels.sum().item())
n_markets = len(set(dataset.groups.tolist())) if hasattr(dataset.groups, 'tolist') else len(set(dataset.groups))

print(f'Snapshots: {n:,}')
print(f'Mercados únicos: {n_markets:,}')
print(f'Positivos (YES): {n_pos:,}  ({n_pos / n:.1%})')
print(f'Features numéricas: {dataset.numerical.shape[1]}')
print(f'Embedding de texto: {dataset.text_emb.shape[1]}')
print(f'Categorías únicas: {len(np.unique(dataset.categories.numpy()))}')

Snapshots: 39,545
Mercados únicos: 13,373
Positivos (YES): 12,357  (31.2%)
Features numéricas: 25
Embedding de texto: 384
Categorías únicas: 8


## 2. Split temporal agrupado por market_id

Mismo split que el resto del pipeline: `temporal_grouped`, sin mezclar snapshots del mismo mercado entre train/val/test.

In [3]:
with open(SAVE_DIR / 'run_config.json') as f:
    run_config = json.load(f)

split_cfg = run_config['split']
print(f"Estrategia       : {split_cfg['strategy']}")
print(f"Train            : {split_cfg['train_size']:,}")
print(f"Validation       : {split_cfg['val_size']:,}")
print(f"Test             : {split_cfg['test_size']:,}")
print(f"Seed             : {split_cfg['seed']}")
print(f"Train pos rate   : {split_cfg['class_balance']['train_positive_rate']:.2%}")
print(f"Val pos rate     : {split_cfg['class_balance']['val_positive_rate']:.2%}")
print(f"Test pos rate    : {split_cfg['class_balance']['test_positive_rate']:.2%}")

Estrategia       : temporal_grouped
Train            : 27,683
Validation       : 5,935
Test             : 5,927
Seed             : 42
Train pos rate   : 31.86%
Val pos rate     : 41.75%
Test pos rate    : 17.85%


## 3. Búsqueda de hiperparámetros

Se prueban variantes de `CatBoostRegressor` sobre residual. La selección sigue la misma lógica global del repo:
`top_k_pnl > 0 AND beats_market_prob → top_k_pnl → brier → log_loss`.

In [4]:
if RUN_TRAINING:
    result = subprocess.run(
        [sys.executable, '-m', 'src.model.train', '--config', str(ROOT / 'config' / 'config.yaml'), '--only', 'catboost'],
        capture_output=True, text=True, cwd=str(ROOT)
    )
    print(result.stdout[-4000:] if len(result.stdout) > 4000 else result.stdout)
    if result.returncode != 0:
        print('STDERR:', result.stderr[-3000:])

with open(SAVE_DIR / 'training_history.json') as f:
    history = json.load(f)

candidates = history.get('candidates', [])
if candidates:
    rows = []
    for i, c in enumerate(candidates):
        params = c.get('model_params', {})
        val = c.get('validation', {})
        rows.append({
            'Candidato': f'C{i+1}',
            'iters': params.get('iterations'),
            'depth': params.get('depth'),
            'lr': params.get('learning_rate'),
            'l2': params.get('l2_leaf_reg'),
            'min_leaf': params.get('min_data_in_leaf'),
            'bag_temp': params.get('bagging_temperature'),
            'rand_strength': params.get('random_strength'),
            'best_iter': c.get('best_iteration'),
            'calibration': c.get('selected_calibration'),
            'val_brier': val.get('brier'),
            'val_logloss': val.get('log_loss'),
            'top_k_hit': val.get('top_k_hit_rate'),
            'top_k_pnl': val.get('top_k_avg_realized_pnl'),
        })
    df_hp = pd.DataFrame(rows)
    best_cfg = run_config['model']
    df_hp['selected'] = (
        (df_hp['iters'] == best_cfg['iterations']) &
        (df_hp['depth'] == best_cfg['depth']) &
        (df_hp['lr'] == best_cfg['learning_rate']) &
        (df_hp['l2'] == best_cfg['l2_leaf_reg']) &
        (df_hp['min_leaf'] == best_cfg['min_data_in_leaf'])
    )
    display(df_hp.style.format({
        'lr': '{:.3f}',
        'l2': '{:.1f}',
        'bag_temp': '{:.1f}',
        'rand_strength': '{:.1f}',
        'val_brier': '{:.5f}',
        'val_logloss': '{:.5f}',
        'top_k_hit': '{:.0%}',
        'top_k_pnl': '{:.4f}',
    }).apply(lambda _: ['background-color: #c8e6c9' if v else '' for v in df_hp['selected']], axis=0))
else:
    print('No hay histórico de candidatos.')

,Candidato,iters,depth,lr,l2,min_leaf,bag_temp,rand_strength,best_iter,calibration,val_brier,val_logloss,top_k_hit,top_k_pnl,selected
0,C1,500,4,0.030,8.0,40,0.5,1.5,108,isotonic,0.09156,0.26267,92%,0.2635,True
1,C2,700,5,0.040,10.0,60,1.0,2.0,107,isotonic,0.09184,0.26356,79%,0.0935,False
2,C3,700,6,0.050,10.0,50,1.0,1.5,54,isotonic,0.09123,0.26191,90%,0.2462,False
3,C4,900,6,0.030,12.0,80,1.0,2.0,87,isotonic,0.09148,0.26261,89%,0.1317,False
4,C5,500,6,0.070,8.0,70,0.0,1.0,23,isotonic,0.09114,0.26182,76%,0.1203,False
5,C6,800,7,0.030,14.0,100,1.0,3.0,102,isotonic,0.09171,0.26278,84%,0.0984,False


In [5]:
if candidates:
    fig, axes = plt.subplots(1, 3, figsize=(14, 4))
    metric_pairs = [
        ('val_brier', 'Brier (val) ↓'),
        ('val_logloss', 'Log-loss (val) ↓'),
        ('top_k_pnl', 'Top-K PnL (val) ↑'),
    ]
    for ax, (col, label) in zip(axes, metric_pairs):
        colors = ['#0D47A1' if s else '#B0BEC5' for s in df_hp['selected']]
        ax.bar(df_hp['Candidato'], df_hp[col], color=colors, edgecolor='white')
        ax.set_title(label, fontsize=11)
        for bar, val in zip(ax.patches, df_hp[col]):
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height(), f'{val:.4f}', ha='center', va='bottom', fontsize=8)
    fig.suptitle('Búsqueda de Hiperparámetros — CatBoost residual (validación)', fontsize=12)
    plt.tight_layout()
    plt.savefig(ROOT / 'figures' / 'catboost_hp_search.png', dpi=150, bbox_inches='tight')
    plt.show()

/var/folders/dv/l82lzhjj64v3xgj4xs_hdqn80000gn/T/ipykernel_25522/1632625278.py:17: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 4. Calibración de probabilidades

Para este modelo, la calibración se elige entre `identity`, `platt` e `isotonic`.
A diferencia del GBDT, aquí la selección final entre calibradores elegibles prioriza **no matar la cola económica** en validación.

In [6]:
cal_info = run_config.get('calibration', {})
selected_cal = cal_info.get('selected', 'identity')
candidates_cal = cal_info.get('candidates', {})

print(f'Calibración seleccionada: {selected_cal}')
print()

if candidates_cal:
    cal_rows = []
    for name, metrics in candidates_cal.items():
        cal_rows.append({
            'Método': name,
            'Brier (val)': metrics.get('brier'),
            'Log-loss (val)': metrics.get('log_loss'),
            'ECE (val)': metrics.get('ece'),
            'selected': name == selected_cal,
        })
    df_cal = pd.DataFrame(cal_rows)
    display(df_cal.drop('selected', axis=1).style.format({
        'Brier (val)': '{:.5f}',
        'Log-loss (val)': '{:.5f}',
        'ECE (val)': '{:.5f}',
    }).apply(lambda _: ['background-color: #c8e6c9' if v else '' for v in df_cal['selected']], axis=0))

Calibración seleccionada: isotonic



,Método,Brier (val),Log-loss (val),ECE (val)
0,identity,0.09431,0.27611,0.02187
1,platt,0.09590,0.28795,0.03567
2,isotonic,0.09156,0.26267,0.00000


## 5. Métricas en test: CatBoost vs Mercado

In [7]:
with open(SAVE_DIR / 'test_metrics.json') as f:
    test_metrics = json.load(f)

tm = test_metrics['test']
vm = test_metrics['validation']
beaten = test_metrics.get('market_baseline_beaten', {})

raw = tm['raw_metrics']
cal = tm['calibrated_metrics']
mkt = tm['market_baseline_metrics']
res = tm.get('residual_metrics', {})
ev = tm['ev_metrics']

summary = pd.DataFrame([
    {'Métrica': 'Brier ↓', 'CatBoost': cal['brier'], 'Mercado': mkt['brier'], 'Gana CatBoost': beaten.get('brier', False)},
    {'Métrica': 'Log-loss ↓', 'CatBoost': cal['log_loss'], 'Mercado': mkt['log_loss'], 'Gana CatBoost': beaten.get('log_loss', False)},
    {'Métrica': 'ROC-AUC ↑', 'CatBoost': cal['roc_auc'], 'Mercado': mkt['roc_auc'], 'Gana CatBoost': cal['roc_auc'] >= mkt['roc_auc']},
    {'Métrica': 'PR-AUC ↑', 'CatBoost': cal['pr_auc'], 'Mercado': mkt['pr_auc'], 'Gana CatBoost': cal['pr_auc'] >= mkt['pr_auc']},
    {'Métrica': 'ECE ↓', 'CatBoost': cal['ece'], 'Mercado': mkt['ece'], 'Gana CatBoost': cal['ece'] <= mkt['ece']},
])

display(summary.style.format({'CatBoost': '{:.5f}', 'Mercado': '{:.5f}'}).apply(
    lambda _: ['background-color: #c8e6c9' if v else 'background-color: #ffcdd2' for v in summary['Gana CatBoost']], axis=0
))

print(f"Residual MAE (test): {res.get('mae_residual', np.nan):.5f}")
print(f"Top-{ev.get('top_k', 'K')} hit rate     : {ev.get('top_k_hit_rate', 'N/A')}")
print(f"Top-{ev.get('top_k', 'K')} avg PnL     : {ev.get('top_k_avg_realized_pnl', 'N/A'):.4f}")
print(f"Top-{ev.get('top_k', 'K')} avg pred EV : {ev.get('top_k_avg_predicted_ev', 'N/A'):.4f}")

,Métrica,CatBoost,Mercado,Gana CatBoost
0,Brier ↓,0.00421,0.00618,True
1,Log-loss ↓,0.01431,0.02210,True
2,ROC-AUC ↑,0.99993,0.99996,False
3,PR-AUC ↑,0.99940,0.99984,False
4,ECE ↓,0.00779,0.01483,True


Residual MAE (test): 0.08138
Top-100 hit rate     : 0.99
Top-100 avg PnL     : 0.1129
Top-100 avg pred EV : 0.0428


In [8]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

comparisons = [
    ('Brier ↓', cal['brier'], mkt['brier']),
    ('Log-loss ↓', cal['log_loss'], mkt['log_loss']),
    ('ECE ↓', cal['ece'], mkt['ece']),
]
for ax, (title, cb_val, mkt_val) in zip(axes, comparisons):
    ax.bar(['CatBoost', 'Mercado'], [cb_val, mkt_val], color=['#0D47A1', '#B0BEC5'], edgecolor='white')
    ax.set_title(title)
    for x, y in zip(['CatBoost', 'Mercado'], [cb_val, mkt_val]):
        ax.text(x, y, f'{y:.4f}', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.savefig(ROOT / 'figures' / 'catboost_vs_market_test.png', dpi=150, bbox_inches='tight')
plt.show()

/var/folders/dv/l82lzhjj64v3xgj4xs_hdqn80000gn/T/ipykernel_25522/1671004102.py:16: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 6. Curvas ROC y Precision-Recall

Se reconstruyen las predicciones finales del test set usando el residual predicho y la calibración guardada.

In [9]:
training_cfg = cfg.get('training', {})
seed = int(training_cfg.get('seed', 42))
val_split = float(training_cfg.get('val_split', 0.15))
test_split = float(training_cfg.get('test_split', 0.15))

split = build_dataset_split(
    n_samples=len(dataset),
    labels=dataset.labels.numpy(),
    timestamps=dataset.timestamps,
    groups=dataset.groups,
    val_split=val_split,
    test_split=test_split,
    strategy='temporal_grouped',
    seed=seed,
)
test_idx = np.asarray(split.test_indices, dtype=np.int64)

model_obj = CatBoostRegressor()
model_obj.load_model(SAVE_DIR / 'catboost_model.cbm')
calibrator = ProbabilityCalibrator.load(SAVE_DIR / 'calibration')

numerical = dataset.numerical.numpy()
categories = dataset.categories.numpy()
text_embs = dataset.text_emb.numpy()
labels_all = dataset.labels.numpy().astype(np.int64)

X_test = build_catboost_matrix(numerical[test_idx], categories[test_idx], text_embs[test_idx])
num_dim = numerical.shape[1]
pool_test = Pool(X_test, cat_features=[num_dim])

raw_residual = np.asarray(model_obj.predict(pool_test), dtype=np.float32)
mkt_prices = np.asarray(dataset.snapshot_prices, dtype=np.float32)[test_idx]
y_test = labels_all[test_idx]
y_raw = clip_probabilities_from_residual(raw_residual, mkt_prices)
y_score = calibrator.predict_proba(y_raw)

print(f'Test samples: {len(y_test):,}  |  Positivos: {y_test.sum():,}  ({y_test.mean():.1%})')

Test samples: 5,927  |  Positivos: 1,058  (17.9%)


In [10]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

ax = axes[0]
for scores, label, color in [
    (y_score, 'CatBoost', '#0D47A1'),
    (mkt_prices, 'Mercado', '#90A4AE'),
]:
    fpr, tpr, _ = roc_curve(y_test, scores)
    roc_auc_val = auc(fpr, tpr)
    ax.plot(fpr, tpr, label=f'{label} (AUC={roc_auc_val:.4f})', color=color, lw=2)
ax.plot([0, 1], [0, 1], 'k--', lw=1)
ax.set_xlabel('FPR')
ax.set_ylabel('TPR')
ax.set_title('Curva ROC — Test')
ax.legend(fontsize=9)

ax = axes[1]
for scores, label, color in [
    (y_score, 'CatBoost', '#0D47A1'),
    (mkt_prices, 'Mercado', '#90A4AE'),
]:
    prec, rec, _ = precision_recall_curve(y_test, scores)
    pr_auc_val = auc(rec, prec)
    ax.plot(rec, prec, label=f'{label} (AUC={pr_auc_val:.4f})', color=color, lw=2)
ax.set_xlabel('Recall')
ax.set_ylabel('Precision')
ax.set_title('Curva Precision-Recall — Test')
ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig(ROOT / 'figures' / 'catboost_roc_pr.png', dpi=150, bbox_inches='tight')
plt.show()

/var/folders/dv/l82lzhjj64v3xgj4xs_hdqn80000gn/T/ipykernel_25522/2021671505.py:32: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 7. Distribución de scores en test

In [11]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

ax = axes[0]
ax.hist(y_score[y_test == 0], bins=50, alpha=0.6, label='No resolvió Yes', color='#EF9A9A', density=True)
ax.hist(y_score[y_test == 1], bins=50, alpha=0.6, label='Resolvió Yes', color='#0D47A1', density=True)
ax.set_xlabel('p_yes final de CatBoost')
ax.set_ylabel('Densidad')
ax.set_title('Distribución de scores — Test')
ax.legend()

ax = axes[1]
ax.scatter(mkt_prices, y_score, c=y_test, cmap='RdYlGn', alpha=0.3, s=8)
ax.plot([0, 1], [0, 1], 'k--', lw=1, label='CatBoost = Mercado')
ax.set_xlabel('price_yes (mercado)')
ax.set_ylabel('p_yes (CatBoost)')
ax.set_title('CatBoost vs Precio de Mercado')
ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig(ROOT / 'figures' / 'catboost_score_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

/var/folders/dv/l82lzhjj64v3xgj4xs_hdqn80000gn/T/ipykernel_25522/3047185133.py:21: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 8. Importancia de features

CatBoost sí expone importancia nativa. Para que el gráfico sea legible, agregamos la categoría como bloque y todo el embedding de texto como otro bloque.

In [12]:
metadata = json.loads((PROCESSED / 'metadata.json').read_text())
feature_names_num = metadata['tabular_feature_names']
num_dim = len(feature_names_num)

importances = model_obj.get_feature_importance(pool_test, type='FeatureImportance')

imp_num = importances[:num_dim]
imp_cat = float(importances[num_dim])
imp_text = float(importances[num_dim + 1:].sum())

all_names = feature_names_num + ['category_id', f'text_embedding ({text_embs.shape[1]}D)']
all_imps = list(imp_num) + [imp_cat, imp_text]

df_imp = pd.DataFrame({'feature': all_names, 'importance': all_imps})
df_imp = df_imp.sort_values('importance', ascending=True).tail(20)

fig, ax = plt.subplots(figsize=(8, 6))
colors = ['#F9A825' if n in ['snapshot_price_yes', 'neg_risk'] else '#1565C0' for n in df_imp['feature']]
ax.barh(df_imp['feature'], df_imp['importance'], color=colors, edgecolor='white')
ax.set_xlabel('Feature importance nativa')
ax.set_title('Top-20 Features — CatBoost residual', fontsize=12)
plt.tight_layout()
plt.savefig(ROOT / 'figures' / 'catboost_feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

/var/folders/dv/l82lzhjj64v3xgj4xs_hdqn80000gn/T/ipykernel_25522/2354996183.py:24: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 9. Análisis económico: EV y Top-K por horizonte

In [13]:
by_horizon = tm.get('by_horizon', {})

bucket_rows = []
for bucket_name, bucket_data in by_horizon.items():
    prob_m = bucket_data.get('probability_metrics', {})
    mkt_m = bucket_data.get('market_baseline_metrics', {})
    ev_m = bucket_data.get('ev_metrics', {})
    bucket_rows.append({
        'Horizonte': bucket_name.replace('_', ' '),
        'n': bucket_data.get('count', 0),
        'CatBoost Brier': prob_m.get('brier'),
        'Mkt Brier': mkt_m.get('brier'),
        'CatBoost ROC-AUC': prob_m.get('roc_auc'),
        'Top-K hit rate': ev_m.get('top_k_hit_rate'),
        'Top-K PnL': ev_m.get('top_k_avg_realized_pnl'),
    })

df_bkt = pd.DataFrame(bucket_rows)
display(df_bkt.style.format({
    'CatBoost Brier': '{:.4f}',
    'Mkt Brier': '{:.4f}',
    'CatBoost ROC-AUC': '{:.4f}',
    'Top-K hit rate': '{:.0%}',
    'Top-K PnL': '{:.4f}',
}))

,Horizonte,n,CatBoost Brier,Mkt Brier,CatBoost ROC-AUC,Top-K hit rate,Top-K PnL
0,short 1 3d,2380,0.0032,0.0052,1.0000,100%,0.0255
1,medium 4 14d,2380,0.0032,0.0052,1.0000,100%,0.0264
2,long 15plus,1167,0.0085,0.0101,0.9997,99%,0.0692


In [14]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

ax = axes[0]
x = np.arange(len(df_bkt))
w = 0.35
ax.bar(x - w/2, df_bkt['CatBoost Brier'], w, label='CatBoost', color='#0D47A1')
ax.bar(x + w/2, df_bkt['Mkt Brier'], w, label='Mercado', color='#90A4AE')
ax.set_xticks(x)
ax.set_xticklabels(df_bkt['Horizonte'], rotation=15)
ax.set_title('Brier por horizonte')
ax.legend()

ax = axes[1]
colors_pnl = ['#0D47A1' if v > 0 else '#EF9A9A' for v in df_bkt['Top-K PnL']]
ax.bar(df_bkt['Horizonte'], df_bkt['Top-K PnL'], color=colors_pnl, edgecolor='white')
ax.axhline(0, color='black', lw=1, ls='--')
ax.set_title('Top-K PnL promedio por horizonte')
ax.set_ylabel('PnL')
for bar, val in zip(ax.patches, df_bkt['Top-K PnL']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.003, f'{val:.3f}', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.savefig(ROOT / 'figures' / 'catboost_by_horizon.png', dpi=150, bbox_inches='tight')
plt.show()

/var/folders/dv/l82lzhjj64v3xgj4xs_hdqn80000gn/T/ipykernel_25522/932794532.py:24: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 10. Resumen final

In [15]:
print('=== CatBoost residual — Resumen ===')
print(
    f"  Modelo: depth={run_config['model']['depth']}, lr={run_config['model']['learning_rate']}, "
    f"iters={run_config['model']['iterations']}, best_iter={run_config['model']['best_iteration']}"
)
print(f"  Calibración: {run_config['calibration']['selected']}")
print()
print('  --- Test metrics finales ---')
print(f"  Brier CatBoost: {cal['brier']:.5f}  |  Mercado: {mkt['brier']:.5f}  {'✓ GANA' if beaten.get('brier') else '✗ pierde'}")
print(f"  Log-loss      : {cal['log_loss']:.5f}  |  Mercado: {mkt['log_loss']:.5f}  {'✓ GANA' if beaten.get('log_loss') else '✗ pierde'}")
print(f"  ROC-AUC       : {cal['roc_auc']:.5f}")
print(f"  PR-AUC        : {cal['pr_auc']:.5f}")
print(f"  Residual MAE  : {res.get('mae_residual', np.nan):.5f}")
print()
print(f"  Top-{ev.get('top_k', 'K')} hit rate : {ev.get('top_k_hit_rate', 'N/A')}")
print(f"  Top-{ev.get('top_k', 'K')} avg PnL : {ev.get('top_k_avg_realized_pnl', 'N/A'):.4f}")
print()
print('Figuras generadas:')
for fig_name in ['catboost_hp_search', 'catboost_vs_market_test', 'catboost_roc_pr', 'catboost_score_distribution', 'catboost_feature_importance', 'catboost_by_horizon']:
    p = ROOT / 'figures' / f'{fig_name}.png'
    print(f"  {'✓' if p.exists() else '?'} figures/{fig_name}.png")

=== CatBoost residual — Resumen ===
  Modelo: depth=4, lr=0.03, iters=500, best_iter=108
  Calibración: isotonic

  --- Test metrics finales ---
  Brier CatBoost: 0.00421  |  Mercado: 0.00618  ✓ GANA
  Log-loss      : 0.01431  |  Mercado: 0.02210  ✓ GANA
  ROC-AUC       : 0.99993
  PR-AUC        : 0.99940
  Residual MAE  : 0.08138

  Top-100 hit rate : 0.99
  Top-100 avg PnL : 0.1129

Figuras generadas:
  ✓ figures/catboost_hp_search.png
  ✓ figures/catboost_vs_market_test.png
  ✓ figures/catboost_roc_pr.png
  ✓ figures/catboost_score_distribution.png
  ✓ figures/catboost_feature_importance.png
  ✓ figures/catboost_by_horizon.png
